# 다구간 소득 모델 — 통일본 (4개 모델 통일 완료)

> 3조 후반부 · 2026-06-04 · `fare_simulation.ipynb`(원본) 기반
>
> 원본 다구간 소득 모델을 나머지 3개 모델(단일-연령, 단일-소득, 다구간-연령)과 **완전히 동일한 환경**으로 통일했다.

| 항목 | 원본 | 통일본 |
|---|---|---|
| 목적함수 | NSB = 복지편익 + 운임수익 - **수송원가** | **NSB = 복지편익 + 운임수익** (수송원가 제거) |
| 복지편익 단가 | 216,280원/인·연 | **탑승당 KOTI κ**(330+939, 2014→CPI) |
| 인플레이션 | 2.5% 균일 복리 | **공통 CPI**(2020=100, 2026후 2.0%) |
| 행동반응 | 활동보존율(탑승 고정) | **ε(-0.3, -0.05)** + 필수/재량 분해 |
| 추정치 | subway_use_rate·activity_sensitivity | **전면 삭제** (탑승 분배=소득분위 인구비중) |
| 출력 | 원(raw) | 억원/연 |

**소득 구간**: 중위소득 분위(50%미만/50-100/100-150/150%+). *단일경계 소득 모델은 기초연금 수급 기준이므로 기준 자체가 다름(설계상 의도).*
**핵심 매핑**: 부담률 b = 단일경계 ρ=f/f₀ (부과요금 비율).


In [1]:
from __future__ import annotations
import csv, math
from dataclasses import dataclass, field
from pathlib import Path
import pandas as pd, numpy as np

# ===== 공통 CPI 시계열 (4개 모델 동일) =====
CPI_2020_BASE = {2014:94.196, 2015:94.861, 2016:95.783, 2017:97.645, 2018:99.086,
                 2019:99.466, 2020:100.0, 2021:102.5, 2022:107.72, 2023:111.59,
                 2024:114.18, 2025:117.4}
_CPI_LAST = max(CPI_2020_BASE); FUTURE_INFL = 0.02
def cpi_factor(fr: int, to: int) -> float:
    def g(y):
        if y in CPI_2020_BASE: return CPI_2020_BASE[y]
        if y > _CPI_LAST: return CPI_2020_BASE[_CPI_LAST]*((1+FUTURE_INFL)**(y-_CPI_LAST))
        return CPI_2020_BASE[2014]/((1+FUTURE_INFL)**(2014-y))
    return g(to)/g(fr)

In [2]:
# ===== 데이터 클래스 =====
@dataclass(frozen=True)
class IncomeGroup:
    name: str; income_range: str; score: int
    population_share: float   # 소득분위별 노인 인구 비중 (탑승 분배용, 실데이터)
    # [통일] 근거 없는 추정치 제거: subway_use_rate·activity_sensitivity 삭제

@dataclass(frozen=True)
class YearInput:
    year: int; predicted_free_rides: int

@dataclass(frozen=True)
class PolicyParams:
    base_fare_2026: float = 1550.0
    min_burden: float = 0.0
    max_burden: float = 1.0
    kappa_freq_2014: float = 330.0   # 의료230 + 사회100 (매 탑승)
    kappa_act_2014:  float = 939.0   # 우울322 + 자살617 (외출 자체 포기분)
    discretionary_share: float = 0.62   # sigma
    epsilon_intensive: float = -0.3
    epsilon_extensive: float = -0.05

@dataclass(frozen=True)
class PolicyConstraints:
    max_burden_by_group: dict; min_burden_by_group: dict

# 소득 4분위 (중위소득 기준). population_share = 분위별 노인 비중(원본 fare_simulation 값)
DEFAULT_INCOME_GROUPS = [
    IncomeGroup("low",          "50% 미만",       1, 851/3080),
    IncomeGroup("lower_middle", "50~100% 미만",   2, 1023/3080),
    IncomeGroup("upper_middle", "100~150% 미만",  3, 587/3080),
    IncomeGroup("high",         "150% 이상",      4, 619/3080),
]
DEFAULT_CONSTRAINTS = PolicyConstraints(
    max_burden_by_group={"low":0.10,"lower_middle":0.35,"upper_middle":0.65,"high":1.00},
    min_burden_by_group={"low":0.00,"lower_middle":0.10,"upper_middle":0.30,"high":0.70},
)

In [3]:
# ===== 연산 로직 (세민 연령 통일본과 동일 구조) =====
def logistic_burden(score, alpha, beta): return 1/(1+math.exp(-(alpha+beta*score)))
def clipped(v, lo, hi): return max(lo, min(hi, v))

def dynamic_income_weights(groups):
    # [통일] 추정치(이용률) 없이 소득분위 인구비중으로만 탑승 분배 (분위 간 1인당 탑승 동일 가정)
    tot = sum(g.population_share for g in groups)
    return {g.name: g.population_share/tot for g in groups}

def burden_rates_by_group(groups, params, alpha, beta):
    return {g.name: clipped(logistic_burden(g.score,alpha,beta), params.min_burden, params.max_burden) for g in groups}

def satisfies_constraints(brs, cons):
    for g,br in brs.items():
        if br < cons.min_burden_by_group.get(g,0.0) or br > cons.max_burden_by_group.get(g,1.0): return False
    return True

# ===== [통일] 행동반응: 단일경계 ε 방식 (부담률 b = rho) =====
def behavior_split(group_rides, burden_rate, params):
    rho = burden_rate
    sigma = params.discretionary_share
    trips_disc = group_rides * sigma
    d_E = clipped(abs(params.epsilon_extensive) * rho, 0.0, 1.0)   # 외출 포기율
    m_I = max(0.0, 1.0 + params.epsilon_intensive * rho)           # 빈도 배수
    forgone_ext = trips_disc * d_E
    forgone_int = trips_disc * (1.0 - d_E) * (1.0 - m_I)
    forgone_dis = forgone_ext + forgone_int
    kept = group_rides - forgone_dis
    return kept, forgone_dis, forgone_ext

def evaluate_policy(yi: YearInput, groups, params: PolicyParams, alpha, beta):
    y = yi.year
    fare = params.base_fare_2026 * cpi_factor(2026, y)
    kf = params.kappa_freq_2014 * cpi_factor(2014, y)
    ka = params.kappa_act_2014  * cpi_factor(2014, y)
    rw = dynamic_income_weights(groups)
    brs = burden_rates_by_group(groups, params, alpha, beta)
    rows = []
    for g in groups:
        b = brs[g.name]
        group_rides = yi.predicted_free_rides * rw[g.name]
        kept, forgone_dis, forgone_ext = behavior_split(group_rides, b, params)
        fare_revenue = kept * fare * b                          # 운임수익
        welfare_baseline = group_rides * kf
        welfare_loss = forgone_dis * kf + forgone_ext * ka
        welfare_total = welfare_baseline - welfare_loss         # 복지편익(수준)
        net_social_benefit = welfare_total + fare_revenue       # 목적함수 NSB
        rows.append({"year":y,"group":g.name,"income_range":g.income_range,
                     "burden_rate":b,
                     "trip_reduction":(forgone_dis/group_rides if group_rides else 0.0),
                     "predicted_rides":round(group_rides),"fare":fare,
                     "welfare_total":welfare_total,"fare_revenue":fare_revenue,
                     "net_social_benefit":net_social_benefit,"alpha":alpha,"beta":beta})
    return rows

def summarize(rows):
    return {"year":int(rows[0]["year"]),
            "welfare_total":sum(r["welfare_total"] for r in rows),
            "fare_revenue":sum(r["fare_revenue"] for r in rows),
            "net_social_benefit":sum(r["net_social_benefit"] for r in rows),
            "alpha":float(rows[0]["alpha"]),"beta":float(rows[0]["beta"])}

def frange(a,b,s):
    c=a
    while c<=b+1e-9: yield round(c,10); c+=s

def optimize_policy(year_inputs, groups, params, cons=DEFAULT_CONSTRAINTS):
    details, summaries = [], []
    alphas, betas = list(frange(-5.0,1.0,0.25)), list(frange(0.25,2.5,0.25))
    for yi in year_inputs:
        best_d, best_s = None, None
        for a in alphas:
            for b in betas:
                if not satisfies_constraints(burden_rates_by_group(groups,params,a,b), cons): continue
                cd = evaluate_policy(yi, groups, params, a, b); cs = summarize(cd)
                if best_s is None or cs["net_social_benefit"] > best_s["net_social_benefit"]:
                    best_d, best_s = cd, cs
        if best_d is None: raise RuntimeError(f"{yi.year}: 실현 가능한 정책 없음")
        details.extend(best_d); summaries.append(best_s)
    return details, summaries

In [4]:
# ===== 억원/연 포맷 출력 =====
def print_report_eok(year, summaries, details):
    s = next((r for r in summaries if r["year"]==year), None)
    if not s: print(f"{year}년 데이터 없음"); return
    print(f"\n===== {year}년 소득구간별 최적 정책 (통일본) =====")
    print(f"  복지편익(수준):  {s['welfare_total']/1e8:>10,.0f} 억원/연")
    print(f"  운임수익:        {s['fare_revenue']/1e8:>10,.0f} 억원/연")
    print(f"  순사회편익(NSB): {s['net_social_benefit']/1e8:>10,.0f} 억원/연  (= 복지편익 + 운임수익)")
    print(f"  최적 alpha/beta: {s['alpha']:.2f} / {s['beta']:.2f}")
    print(f"  {'소득구간':<14}{'부담률':>8}{'탑승감소':>9}")
    for r in [d for d in details if d['year']==year]:
        print(f"  {r['income_range']:<14}{r['burden_rate']*100:>7.1f}%{r['trip_reduction']*100:>8.1f}%")

print("다구간 소득 통일본 모듈 로드 완료.")

다구간 소득 통일본 모듈 로드 완료.


In [5]:
# ===== 데이터 로더 (병철 최종예측 무임수요) =====
from pathlib import Path
import pandas as pd, warnings
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "dataset").exists():
    ROOT = ROOT.parent
RIDES = ROOT / "dataset" / "processed" / "서울시_2026_2050_무임승차_노인수요_최종_병철v2.csv"

def load_rides(csv_path=RIDES):
    df = pd.read_csv(csv_path)
    col = "final_pred" if "final_pred" in df.columns else "연간_총_최종수요"
    return {int(r["year"]): int(round(float(r[col]))) for _, r in df.iterrows()}

def generate_year_inputs(rides_path=RIDES):
    rides_map = load_rides(rides_path)
    return [YearInput(y, rides_map[y]) for y in range(2026, 2051) if y in rides_map]

In [6]:
# ===== 2026~2050 실행 =====
year_inputs = generate_year_inputs()
print(f"연도입력 {len(year_inputs)}개 로드 ({year_inputs[0].year}~{year_inputs[-1].year})")
details, summaries = optimize_policy(year_inputs, DEFAULT_INCOME_GROUPS, PolicyParams())
for y in (2026, 2035, 2050):
    print_report_eok(y, summaries, details)

연도입력 25개 로드 (2026~2050)

===== 2026년 소득구간별 최적 정책 (통일본) =====
  복지편익(수준):       1,005 억원/연
  운임수익:             1,489 억원/연
  순사회편익(NSB):      2,494 억원/연  (= 복지편익 + 운임수익)
  최적 alpha/beta: -4.00 / 1.50
  소득구간               부담률     탑승감소
  50% 미만            7.6%     1.6%
  50~100% 미만       26.9%     5.8%
  100~150% 미만      62.2%    13.1%
  150% 이상          88.1%    18.4%

===== 2035년 소득구간별 최적 정책 (통일본) =====
  복지편익(수준):       1,500 억원/연
  운임수익:             2,222 억원/연
  순사회편익(NSB):      3,722 억원/연  (= 복지편익 + 운임수익)
  최적 alpha/beta: -4.00 / 1.50
  소득구간               부담률     탑승감소
  50% 미만            7.6%     1.6%
  50~100% 미만       26.9%     5.8%
  100~150% 미만      62.2%    13.1%
  150% 이상          88.1%    18.4%

===== 2050년 소득구간별 최적 정책 (통일본) =====
  복지편익(수준):       2,927 억원/연
  운임수익:             4,336 억원/연
  순사회편익(NSB):      7,263 억원/연  (= 복지편익 + 운임수익)
  최적 alpha/beta: -4.00 / 1.50
  소득구간               부담률     탑승감소
  50% 미만            7.6%     1.6%
  50~100% 미만       26.9%     5.8%
  100~150% 미

---
### 통일 결과
이 노트북으로 **4개 모델(단일-연령 / 단일-소득 / 다구간-연령 / 다구간-소득)이 동일한 환경**(CPI·κ·ε·NSB·수송원가 제외·추정치 제거)을 공유한다. 유일한 차이는 **정책 구조**(경계 단일 vs 다구간, 기준 연령 vs 소득)뿐이다.

**주의 — 소득 기준의 차이**: 단일경계 소득 모델은 **기초연금/기초생활 수급**(자치구별 수급률) 기준, 본 다구간 소득 모델은 **중위소득 분위** 기준이다. 두 '소득' 모델은 기준 자체가 달라 직접 비교 시 이 점을 명시해야 한다.

**남은 작업**: 4개 모델 NSB를 한 표/그래프로 비교 (run_unified_2026_2050.py 확장).
